# Tiered KV H100 RunPod Drop-In

This notebook expects `runpod_h100_dropin/` and `tiered_kv/` to be sibling folders. It uses the bundled tuned policy config in `best_policy_config.json`. The default profile is `h100-safe`, which avoids the previous 32K perplexity OOM by testing perplexity up to 16K first.

In [ ]:
# Setup. Run this first, then restart the kernel if packages were installed.
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

%pip install -q -r requirements-runpod.txt

import sys
print('python:', sys.version)
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('PYTORCH_CUDA_ALLOC_CONF:', os.environ.get('PYTORCH_CUDA_ALLOC_CONF'))


In [ ]:
import json
import os
import sys
from pathlib import Path

import torch

DROPIN = Path.cwd().resolve()
PARENT = DROPIN.parent
for path in (DROPIN, PARENT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from run_h100_benchmark import get_profile, run_benchmark, resolve_policy_config, summarize

PROFILE_NAME = os.environ.get('PROFILE_NAME', 'h100-safe')
MODEL_ID = os.environ.get('MODEL_ID') or None
PROFILE = get_profile(PROFILE_NAME, model_id=MODEL_ID)
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
POLICY_CONFIG_FILE = Path(os.environ.get('POLICY_CONFIG_FILE', 'best_policy_config.json'))
if not POLICY_CONFIG_FILE.exists():
    raise FileNotFoundError(f'Missing policy config: {POLICY_CONFIG_FILE}')
POLICY_CONFIG = resolve_policy_config(json.loads(POLICY_CONFIG_FILE.read_text()))

print('dropin:', DROPIN)
print('sibling tiered_kv exists:', (PARENT / 'tiered_kv').exists())
print('profile:', PROFILE)
print('device:', DEVICE)
print('policy_config_file:', POLICY_CONFIG_FILE)
print(json.dumps(POLICY_CONFIG, indent=2))
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## Main Benchmark

Runs recall, perplexity, measured KV-cache memory, and throughput. Use a fresh kernel before rerunning this cell.

In [ ]:
result = run_benchmark(
    PROFILE,
    device=DEVICE,
    policy_config=POLICY_CONFIG,
    run_recall=True,
    run_perplexity=True,
    run_niah=False,
    run_memory=True,
    output_dir=Path('results/runpod_h100'),
)
summary = summarize(result)
summary_path = Path(result['output_path']).with_name(Path(result['output_path']).stem + '_summary.json')
summary_path.write_text(json.dumps(summary, indent=2))
print('saved:', result.get('output_path'))
print('summary:', summary_path)
print(json.dumps(result['sanity'], indent=2))
print(json.dumps(summary, indent=2))


## Tables

In [ ]:
import pandas as pd

recall_rows = []
for group in result.get('recall', []):
    for length, acc, ok, total in zip(group['lengths'], group['accuracy'], group['n_correct'], group['n_total']):
        recall_rows.append({'strategy': group['strategy'], 'context_len': length, 'accuracy': acc, 'n_correct': ok, 'n_total': total})
recall_df = pd.DataFrame(recall_rows)
display(recall_df)

ppl_rows = []
for group in result.get('perplexity', []):
    for row in group['rows']:
        ppl_rows.append({'strategy': group['strategy'], **row})
ppl_df = pd.DataFrame(ppl_rows)
display(ppl_df)

mem_df = pd.DataFrame(result.get('memory', []))
thru_df = pd.DataFrame(result.get('throughput', []))
if not mem_df.empty:
    baseline_bytes = float(mem_df.loc[mem_df['strategy'] == 'FP16 baseline', 'cache_bytes_empirical'].iloc[0])
    mem_df['cache_MB'] = mem_df['cache_bytes_empirical'] / 1e6
    mem_df['ratio_to_baseline'] = mem_df['cache_bytes_empirical'] / baseline_bytes
    display(mem_df[['strategy', 'seq_len', 'cache_MB', 'ratio_to_baseline', 'peak_alloc_bytes', 'peak_reserved_bytes']])
if not thru_df.empty:
    display(thru_df[['strategy', 'tokens_per_second', 'median_seconds', 'n_new_tokens']])


## Plots

In [ ]:
import matplotlib.pyplot as plt

if not recall_df.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    for name, g in recall_df.groupby('strategy'):
        ax.plot(g['context_len'], g['accuracy'], marker='o', label=name)
    ax.set_xscale('log')
    ax.set_xlabel('context length')
    ax.set_ylabel('recall accuracy')
    ax.set_title('Synthetic key-value recall')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

if not ppl_df.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    for name, g in ppl_df.groupby('strategy'):
        ax.plot(g['context_len'], g['perplexity'], marker='o', label=name)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('context length')
    ax.set_ylabel('perplexity')
    ax.set_title('Sliding-window perplexity')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

if not mem_df.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(mem_df['strategy'], mem_df['ratio_to_baseline'])
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
    ax.set_ylabel('cache bytes / FP16 baseline')
    ax.set_title('Measured KV-cache memory')
    ax.tick_params(axis='x', rotation=20)
    plt.show()


## Optional NIAH

Run this after the main benchmark, preferably from a fresh kernel if memory is tight.

In [ ]:
# Uncomment to run NIAH only.
# import torch
# torch.cuda.empty_cache()
# niah_result = run_benchmark(
#     PROFILE,
#     device=DEVICE,
#     policy_config=POLICY_CONFIG,
#     run_recall=False,
#     run_perplexity=False,
#     run_niah=True,
#     run_memory=False,
#     output_dir=Path('results/runpod_h100_niah'),
# )
# niah_summary = summarize(niah_result)
# print(niah_result.get('output_path'))
# print(json.dumps(niah_summary, indent=2))
